# 1kg_eur — KING parent-offspring exclusion (on Batch)

Full-sib (FS) and parent-offspring (PO) pairs have the same *expected*
additive relatedness (`a_ij ~= 0.5`), so the GRM alone (what
`04_grm_panel_qc.ipynb` -> `06_grm_shards.ipynb` builds) cannot tell them
apart -- both land in the same relatedness bin. What differs is the
IBD-sharing *pattern*: a PO pair shares exactly 1 allele IBD at
(essentially) every locus, so it never shows an IBS0 genotype combination
(both homozygous for opposite alleles) except by genotyping error. A FS pair
shares 0 or 2 alleles at a real fraction of loci, so a nonzero IBS0 rate is
expected. That is the signal [KING](https://www.kingrelatedness.com/)'s
`--related` mode uses to auto-classify pairs within the same kinship band
(`InfType`: Dup/MZ, PO, FS, 2nd, 3rd, UN).

**Both compute steps run as a single `dsub`/Google Batch job:**

1. **thin** the QC'd GRM panel (~1.25M variants, ~63 GB BED) to ~50K
   variants with `plink2 --thin`, then
2. **run `king --related --degree 1`** on the thinned panel.

One job, not two, because step 2's only input is step 1's output and the
expensive part of either step is localizing the 63 GB panel onto the worker
-- two jobs would pay that twice. The notebook VM only ever pulls back the
`.kin0` table (a few MB), never the panel.

**Scope:** identifies PO pairs and writes an exclusion list. **Not yet wired
into the accumulate step** -- `grm_shard_tool` has no per-pair exclusion
option, so this list isn't applied to `08_accumulate.ipynb`'s binned
cross-products yet. See the last section.

**Prerequisites:**
- `04_grm_panel_qc.ipynb` — `1kg_CEUGBR_GRM_QC.{bed,bim,fam}` in `03_grm/grm_input/`
- `king` and `plink2` staged to the bucket (the staging cell below does it)

## Config

In [ ]:
import os, subprocess

PROJECT_ID      = "wb-swift-sprout-7231"
REGION          = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK         = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK      = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
CLOUD_SDK_TAG   = "581.0.0-slim"

WS_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS  = f"{WS_GS}/1kg_eur"

# ── input ─────────────────────────────────────────────────────────────────────
# The QC'd GRM panel from 04_grm_panel_qc.ipynb (MAF > 1%, ~1.25M variants),
# already restricted to the round-2 CEU/GBR-anchored sample set -- the same
# panel 06_grm_shards.ipynb computes the GRM from, so no secondary --keep here.
BED_NAME     = "1kg_CEUGBR_GRM_QC"
GRM_INPUT_GS = f"{R_GS}/03_grm/grm_input"
BIN_GS       = f"{R_GS}/03_grm/bin"        # king + plink2, staged below

# ── output ────────────────────────────────────────────────────────────────────
# The thinned panel gets its own bucket dir, separate from the KING results, so
# a KING-only re-run can take it as an input and skip step 1 (see "Submit").
REL_GS  = f"{R_GS}/04_relatedness"
THIN_GS = f"{REL_GS}/king_thinned"
KING_GS = f"{REL_GS}/king_po_exclusion"
LOG_GS  = f"{REL_GS}/logs"

# <= 10 chars on purpose: dsub truncates the job-name portion of the job-id
# to 10 characters, so a longer name here would make the dstat wildcard below
# silently match nothing.
JOB_NAME       = "r2-king"
THIN_NAME      = f"{BED_NAME}_king_thinned"
KING_OUT_NAME  = f"{THIN_NAME}_king"       # KING's --prefix basename

# ── parameters ────────────────────────────────────────────────────────────────
# Target SNP count for KING specifically, far below the full GRM panel.
# Kinship/IBS0 estimation doesn't need genome-wide density -- a few tens of
# thousands of independent SNPs already gives precise estimates -- and KING's
# runtime is linear in SNP count.
KING_N_SNPS_TARGET = 50_000

# 1st-degree only (PO + FS): that's the ambiguity this notebook exists to
# resolve. Bump to 2 if 2nd-degree pairs become relevant later.
DEGREE = 1

# ── machine sizing ────────────────────────────────────────────────────────────
# A *predefined* machine type on purpose. n1-custom would require memory in
# exact multiples of 256 MB (plus an -ext suffix past 8192 MB/vCPU), and getting
# that wrong makes Batch reject the VM spec outright -- before any task or log
# exists, so it presents as every task failing with no logs at all. Nothing here
# needs a custom shape, so don't invite that failure mode.
#
# n1-highmem-16 = 16 vCPU / 104 GB. Neither step is memory-hungry: plink2
# streams the BED for --make-bed, and KING packs genotypes into two bit-arrays
# (227,561 samples x 788 words x 8 B x 2 ~= 2.9 GB at 50K SNPs). The headroom is
# deliberate anyway -- an OOM wastes the entire 63 GB localization, which is the
# dominant cost of the job. Both steps scale with cores, so vCPUs are the dial
# worth touching if this turns out slow.
MACHINE_TYPE = "n1-highmem-16"

# 63 GB panel in + ~2.9 GB thinned out (227,561 x 50,374 / 4 bytes) + slack.
# Asserted against the panel's real size in the preflight cell below.
DISK_SIZE_GB = 150

for k, v in dict(MACHINE_TYPE=MACHINE_TYPE, DISK_SIZE_GB=DISK_SIZE_GB,
                 KING_N_SNPS_TARGET=KING_N_SNPS_TARGET, DEGREE=DEGREE,
                 THIN_GS=THIN_GS, KING_GS=KING_GS).items():
    print(f"  {k}: {v}")

## Install dsub and patch the wrapper image

dsub hardcodes a `CLOUD_SDK_IMAGE` tag in `providers/google_utils.py` for five
wrapper runnables (localize / log-stream / delocalize); `--image` only sets the
sixth, user-command runnable. Every tag pinned by a released dsub version has
been withdrawn from gcr.io, so patching is mandatory — and it has to happen **in
the same shell as the `dsub` call**, because the patch lives in site-packages
and any `pip install dsub` silently reverts it. dsub >= 0.5.3 is also required:
0.5.2's wrappers shell out to `gsutil`, which current cloud-sdk images no longer
ship, and the resulting failure is indistinguishable from a dead image tag.

In [ ]:
subprocess.run(["bash", "-c", f"""
pip install --quiet --upgrade 'dsub>=0.5.3'
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
echo "dsub: $(dsub --version)"
echo "--- wrapper copy tool (must be 'gcloud storage cp', not 'gsutil') ---"
grep -o 'gcloud storage cp\\|gsutil .*cp' "$DSUB_DIR/providers/google_utils.py" | sort -u
"""], check=True)

## Stage the `king` and `plink2` binaries

The Batch worker cannot fetch these itself: the worker image ships neither
`wget` nor `curl`, and the job runs with no external IP under VPC Service
Controls, so there is no general internet egress either (Private Google Access
covers Google APIs only). Both binaries have to be staged into the bucket from
this VM and localized like any other input.

Each prefers an already-staged copy in GCS over a download, since upstream URLs
rot — this notebook's original `king` URL 404'd. No `set -e`, so a failure on
one binary doesn't hide the state of the other.

In [ ]:
subprocess.run(["bash", "-c", f"""
BIN_DIR="$HOME/bin"; mkdir -p "$BIN_DIR"

# ---- KING ----
if [ -x "$BIN_DIR/king" ]; then
  echo "king:   already present locally"
elif gcloud storage cp "{BIN_GS}/king" "$BIN_DIR/king" 2>/dev/null; then
  chmod +x "$BIN_DIR/king"; echo "king:   copied from {BIN_GS}/king"
elif ( cd /tmp && wget -q -O king.tar.gz "https://www.kingrelatedness.com/Linux-king.tar.gz" && tar -xzf king.tar.gz -C "$BIN_DIR" king ); then
  chmod +x "$BIN_DIR/king"; echo "king:   downloaded"
else
  echo "king:   UNAVAILABLE — current link is on the Download page of kingrelatedness.com"
fi

# ---- plink2 ----
if [ -x "$BIN_DIR/plink2" ]; then
  echo "plink2: already present locally"
elif gcloud storage cp "{BIN_GS}/plink2" "$BIN_DIR/plink2" 2>/dev/null; then
  chmod +x "$BIN_DIR/plink2"; echo "plink2: copied from {BIN_GS}/plink2"
elif ( cd /tmp && wget -q -O plink2.zip "https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip" && unzip -o -q plink2.zip plink2 -d "$BIN_DIR" ); then
  chmod +x "$BIN_DIR/plink2"; echo "plink2: downloaded"
else
  echo "plink2: UNAVAILABLE — current link at cog-genomics.org/plink/2.0/"
fi

echo
# Upload unconditionally rather than skipping when the destination exists: an
# earlier failed copy can leave a truncated object there, and an existence-only
# check would keep reusing it forever.
for b in king plink2; do
  if [ -x "$BIN_DIR/$b" ]; then gcloud storage cp "$BIN_DIR/$b" "{BIN_GS}/$b"; fi
done
gcloud storage ls -l "{BIN_GS}/king" "{BIN_GS}/plink2"
"""], check=True)

## Preflight

dsub does not check `--input` paths before submitting, so a missing object costs
a VM spin-up plus ~60 s of `gcloud storage cp` retries to discover, and the
failure is visible only in the job's main `.log`. Assert everything from here
instead, where a typo or an unrun prerequisite is an instant local error.

In [ ]:
def gs_size(uri):
    """Size in bytes of a single GCS object, or None if it doesn't exist."""
    p = subprocess.run(["gcloud", "storage", "ls", "-l", uri],
                       capture_output=True, text=True)
    if p.returncode != 0:
        return None
    return int(p.stdout.split()[0])

panel = {ext: f"{GRM_INPUT_GS}/{BED_NAME}.{ext}" for ext in ("bed", "bim", "fam")}
bins  = {b: f"{BIN_GS}/{b}" for b in ("king", "plink2")}

sizes, missing = {}, []
for label, uri in {**panel, **bins}.items():
    sizes[label] = gs_size(uri)
    shown = "MISSING" if sizes[label] is None else f"{sizes[label] / 2**30:8.2f} GB"
    print(f"  {label:7s} {shown}  {uri}")
    if sizes[label] is None:
        missing.append(uri)

assert not missing, (
    "missing inputs — run 04_grm_panel_qc.ipynb / the staging cell above:\n  "
    + "\n  ".join(missing)
)

# The worker's disk has to hold the panel, the thinned copy, and some slack.
panel_gb = sum(sizes[e] for e in ("bed", "bim", "fam")) / 2**30
needed_gb = panel_gb * 1.05 + 10
assert DISK_SIZE_GB > needed_gb, (
    f"DISK_SIZE_GB={DISK_SIZE_GB} too small: panel is {panel_gb:.0f} GB, "
    f"need > {needed_gb:.0f} GB"
)
print(f"\nOK — panel {panel_gb:.0f} GB, needs > {needed_gb:.0f} GB, disk is {DISK_SIZE_GB} GB")

## Submit

One task, two steps. The parts that are load-bearing:

- **`plink2 --maj-ref` is required, not cosmetic.** KING reads `.bim` column 5
  (A1) as the *minor* allele — what `plink1.9 --make-bed` writes. plink2 writes
  A1 = ALT instead, and for a plink1 `.bed` input "ALT" is merely whichever
  allele came second in the source file, so A1 is the major allele for a large
  share of variants and KING aborts with *"Too many first alleles as the major
  allele (~15.1%)"*. `--maj-ref` sets REF to the major allele, making
  A1 = ALT = minor. (Safe here because ref alleles from a plink1 `.bed` are
  provisional; a pgen with real ref alleles would need `--maj-ref force`.)
- **`--thin` is a random subsample, and that's adequate here.** The panel isn't
  LD-pruned (it's the QC'd MAF > 1% GRM panel, not `--indep-pairwise`d), but
  random thinning only removes density, it doesn't introduce correlation — so
  the result is still a reasonable approximately-independent SNP set for
  kinship/IBS0 purposes.
- **Step 1 is skipped automatically on a re-run.** If the thinned panel is
  already in `THIN_GS`, the next cell localizes *that* (~2.9 GB) instead of the
  full panel (~63 GB) and goes straight to KING. The choice is made here, client
  side, because dsub localizes inputs before the command runs — the task itself
  cannot opt out of a download it has already been handed. So re-running with a
  different `DEGREE` is cheap; changing `KING_N_SNPS_TARGET` means deleting
  `THIN_GS` first, or the stale thinned panel will just be reused.
- **`--input-recursive`, not three per-file `--input`s.** It keeps `.bed`,
  `.bim` and `.fam` together in one localized directory, so `--bfile` resolves
  without depending on how dsub lays out individually-named inputs. It pulls the
  directory's `_freq.frq` along too, which is tens of MB and harmless.
- **`--related --degree 1`** restricts reporting to pairs KING itself estimates
  as 1st-degree (the Dup/MZ + PO + FS kinship band), labelling each in
  `InfType`. It writes `<prefix>.kin0` for pairs with no family structure; this
  panel's `.fam` carries no real pedigree, so *every* related pair lands there
  rather than being split across `.kin`/`.kin0`.

In [ ]:
# Decide up front whether step 1 can be skipped.
thin_files = {ext: f"{THIN_GS}/{THIN_NAME}.{ext}" for ext in ("bed", "bim", "fam")}
SKIP_THIN = all(gs_size(u) is not None for u in thin_files.values())

if SKIP_THIN:
    print(f"thinned panel present in {THIN_GS}\n  -> localizing it (~3 GB), skipping step 1")
    IO_FLAGS = (f'--input-recursive  THIN_IN="{THIN_GS}" '
                f'--output-recursive KING_DIR="{KING_GS}"')
else:
    print(f"no thinned panel in {THIN_GS}\n  -> localizing the full panel (~63 GB) and thinning")
    IO_FLAGS = (f'--input-recursive  PANEL_DIR="{GRM_INPUT_GS}" '
                f'--output-recursive THIN_DIR="{THIN_GS}" '
                f'--output-recursive KING_DIR="{KING_GS}"')
print(f"\ndsub I/O flags:\n  {IO_FLAGS}")

In [ ]:
subprocess.run(["bash", "-c", f"""
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E "s|cloud-sdk:[0-9]+\\.[0-9]+\\.[0-9]+-slim|cloud-sdk:{CLOUD_SDK_TAG}|g" "$DSUB_DIR/providers/google_utils.py"
find "$DSUB_DIR" -name '*.pyc' -delete
grep CLOUD_SDK_IMAGE "$DSUB_DIR/providers/google_utils.py"

dsub \\
  --provider google-batch --project "{PROJECT_ID}" --regions "{REGION}" \\
  --logging "{LOG_GS}" \\
  --service-account "{SERVICE_ACCOUNT}" \\
  --network "{NETWORK}" --subnetwork "{SUBNETWORK}" --use-private-address \\
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:{CLOUD_SDK_TAG}" \\
  --name "{JOB_NAME}" \\
  --machine-type "{MACHINE_TYPE}" --disk-size "{DISK_SIZE_GB}" \\
  --input KING_BIN="{BIN_GS}/king" \\
  --input PLINK2_BIN="{BIN_GS}/plink2" \\
  {IO_FLAGS} \\
  --env SKIP_THIN="{int(SKIP_THIN)}" \\
  --env BED_NAME="{BED_NAME}" \\
  --env THIN_NAME="{THIN_NAME}" \\
  --env KING_OUT_NAME="{KING_OUT_NAME}" \\
  --env N_TARGET="{KING_N_SNPS_TARGET}" \\
  --env DEGREE="{DEGREE}" \\
  --command '
    set -eo pipefail
    chmod +x "$KING_BIN" "$PLINK2_BIN"
    N_CPUS=$(nproc)

    # dsub creates the localized input dirs, but an --output-recursive dir is
    # only created on delocalization -- plink2 and king need it to exist now.
    mkdir -p "$KING_DIR"

    if [ "$SKIP_THIN" = "1" ]; then
      echo "=== step 1: skipped, reusing the staged thinned panel ==="
      THIN_PREFIX="$THIN_IN/$THIN_NAME"
    else
      echo "=== step 1: thin to ~$N_TARGET SNPs ==="
      mkdir -p "$THIN_DIR"
      THIN_PREFIX="$THIN_DIR/$THIN_NAME"
      N_CURRENT=$(wc -l < "$PANEL_DIR/$BED_NAME.bim")
      # awk rather than python3: makes no assumption about what the worker image ships.
      THIN_P=$(awk -v t="$N_TARGET" -v c="$N_CURRENT" "BEGIN {{ printf \\"%.8f\\", (t < c ? t / c : 1) }}")
      echo "  $N_CURRENT SNPs -> target $N_TARGET (thin p=$THIN_P)"
      "$PLINK2_BIN" \\
        --bfile "$PANEL_DIR/$BED_NAME" \\
        --thin "$THIN_P" \\
        --maj-ref \\
        --make-bed \\
        --threads "$N_CPUS" \\
        --out "$THIN_PREFIX"
    fi
    echo "  thinned SNP count: $(wc -l < "$THIN_PREFIX.bim")"
    echo "  samples:           $(wc -l < "$THIN_PREFIX.fam")"

    echo "=== step 2: king --related --degree $DEGREE on $N_CPUS cpus ==="
    time "$KING_BIN" \\
      -b "$THIN_PREFIX.bed" \\
      --related --degree "$DEGREE" \\
      --cpus "$N_CPUS" \\
      --prefix "$KING_DIR/$KING_OUT_NAME"

    ls -lh "$KING_DIR"
  ' 2>&1 | tee /tmp/king_job.log
"""], check=True)

## Monitor

Matched by job name rather than by a job-id captured from dsub's stdout: that
output doesn't reliably end in a bare job-id, and `tail -1` on it has silently
produced a garbage id that made `dstat` return `[]`. The wildcard also lists
earlier runs of the same job, which is usually what you want here. `check=False`
so a not-yet-visible job doesn't raise.

In [ ]:
subprocess.run(["bash", "-c", f"""
dstat --provider google-batch \\
  --project "{PROJECT_ID}" --location "{REGION}" \\
  --jobs "{JOB_NAME}*" --users '*' --status '*' --full 2>&1 | tail -40
"""], check=False)

On failure, read the **main `<job-id>.log` first**, not the `-stdout`/`-stderr`
pair. Those two only ever capture the user command, so they are empty for *any*
pre-command failure and their emptiness carries no diagnostic information. The
main log captures dsub's own localize/delocalize runnables, and it is what
distinguishes the two failure modes that otherwise look identical from `dstat`
(task reaches RUNNING, exits 1 after ~40–80 s, both streams empty): a missing
`--input` object ends that log with `ERROR: (gcloud.storage.cp) The following
URLs matched no objects`, whereas a purged wrapper image leaves even the main
log empty, because the logging runnable itself can't start.

In [ ]:
subprocess.run(["bash", "-c", f"""
for f in $(gcloud storage ls "{LOG_GS}/**" 2>/dev/null); do
  echo "=========================================== $f"
  gcloud storage cat "$f" 2>/dev/null | tail -25
done
"""], check=False)

## Classify + sanity-check

`InfType` is KING's own call; the IBS0 check re-derives the same distinction
from first principles (PO pairs cluster near `IBS0 ~= 0`, FS pairs sit at a
visibly higher IBS0) as a cross-check that the classification on *this* panel —
thinned, ancestry-filtered, not the dense array data KING is usually pointed at
— looks sane before anything is excluded on the strength of it.

Only the `.kin0` table comes back to the VM; the genotypes stay in the bucket
and on the (now deleted) worker. Note that KING writes no `.kin0` at all if it
finds *zero* pairs at this degree — so a `gcloud storage cp` failure in the next
cell means either the job didn't finish or there are no 1st-degree pairs, which
is worth distinguishing in the job's stdout log before assuming a bug.

In [ ]:
import pandas as pd

LOCAL_DIR = os.path.expanduser("~/scratch_1kg_eur_grm_qc")
os.makedirs(LOCAL_DIR, exist_ok=True)

kin0_gs   = f"{KING_GS}/{KING_OUT_NAME}.kin0"
kin0_path = os.path.join(LOCAL_DIR, os.path.basename(kin0_gs))
subprocess.run(["gcloud", "storage", "cp", kin0_gs, kin0_path], check=True)

kin0 = pd.read_csv(kin0_path, sep=r"\s+")
print(f"{len(kin0)} pairs at degree <= {DEGREE}")
print(kin0["InfType"].value_counts())

print("\nIBS0 by InfType (should show PO near 0, FS clearly higher):")
print(kin0.groupby("InfType")["IBS0"].describe()[["mean", "std", "min", "max"]])

In [ ]:
po_pairs = kin0[kin0["InfType"] == "PO"][["ID1", "ID2", "Kinship", "IBS0"]]
fs_pairs = kin0[kin0["InfType"] == "FS"][["ID1", "ID2", "Kinship", "IBS0"]]
print(f"{len(po_pairs)} PO pairs, {len(fs_pairs)} FS pairs")

for name, df in (("po_pairs_exclude.tsv", po_pairs),
                 ("fs_pairs_confirmed.tsv", fs_pairs)):
    local = os.path.join(LOCAL_DIR, name)
    df.to_csv(local, sep="\t", index=False)
    subprocess.run(["gcloud", "storage", "cp", local, f"{KING_GS}/{name}"], check=True)
    print(f"wrote {len(df):>6} pairs -> {KING_GS}/{name}")

## Follow-up: wiring this into the accumulate step

`po_pairs_exclude.tsv` (this notebook's output) is a **pair-level** exclusion
list, not a sample-level one -- dropping one of the two individuals in a PO pair
would also throw away every *other* pair that individual forms (e.g. with their
own siblings), which isn't what we want here.

`grm_shard_tool accumulate` (`GRM-pairs/grm_bin_sharded/grm_shard_tool.cpp`)
currently bins every pair unconditionally -- there's no `--exclude-pairs` option
to skip specific `(FID, IID)` pairs while accumulating a shard's cross-products.
Applying this exclusion for real (so `h2_FS`/`b2_FS`/`b2_step` in
`09_estimators.ipynb` are computed on PO-pair-free FS bins) needs that added:
read `po_pairs_exclude.tsv` (or a hash set built from it) into `accumulate`, and
skip a pair before it's added to any bin's running sum. Not implemented here --
flagging as the next concrete step rather than silently leaving this notebook's
output unused.

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser("~/repos/AOU-covariance/notebooks/07_relatedness_screen.ipynb")
_gs = f"{R_GS}/notebooks/07_relatedness_screen.ipynb"
subprocess.run(["gcloud", "storage", "cp", _nb, _gs], check=True)
print(f"notebook -> {_gs}")